# Pipeline Farma Help — RAG

Este notebook ejecuta únicamente el flujo del RAG. Toda la implementación vive en ``src/farma_help/rag`` y se consume como librería.

Requisitos previos:
- Instalar dependencias: ``pip install -r requirements.txt``
- Configurar ``.env`` en la raíz del repositorio con ``GEMINI_API_KEY``, ``GEMINI_MODEL`` y ``GEMINI_EMBEDDING_MODEL``.

In [1]:
import os
import sys
from pathlib import Path

RAIZ = Path.cwd().parent

# Permitir importar controles (raíz) y farma_help (src).
sys.path.insert(0, str(RAIZ))
sys.path.insert(0, str(RAIZ / "src"))

from farma_help.configuracion import cargar_entorno, cargar_rutas, variable_entorno

cargar_entorno()

API_KEY = variable_entorno("GEMINI_API_KEY")
MODELO_LLM = variable_entorno("GEMINI_MODEL", "gemini-3.5-flash-lite")
MODELO_EMBEDDING = variable_entorno("GEMINI_EMBEDDING_MODEL", "gemini-embedding-001")

rutas = cargar_rutas()
print("Variables de entorno cargadas.")
print(f"Políticas: {rutas.politicas}")

Variables de entorno cargadas.
Políticas: /home/edison/Desktop/MAESTRIA_DS/ciberseguridad_de_ia/proyecto-github/datos/politicas


## 1. Construir el servicio RAG

Esta celda lee las políticas, crea fragmentos y chunks, genera embeddings con Gemini, construye los índices FAISS por nivel de acceso y carga el reranker y el generador.

In [2]:
from farma_help.rag.servicio import ServicioRAG

rag = ServicioRAG.construir(
    carpeta_politicas=rutas.politicas,
    api_key=API_KEY,
    modelo_llm=MODELO_LLM,
    modelo_embeddings=MODELO_EMBEDDING,
    device="cpu",
    usar_reranking=True,
)
print("✅ Servicio RAG construido.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (694 > 512). Running this sequence through the model will result in indexing errors


✅ Servicio RAG construido.


## 2. Recuperar chunks de prueba

Verifica que la recuperación semántica y los permisos por rol funcionan sin invocar al LLM.

In [7]:
PREGUNTA_PRUEBA = (
    "¿Cuál es el plazo para reclamar un faltante y desde cuándo se cuenta?"
)
ROL_PRUEBA = "operador"

candidatos = rag.recuperar(PREGUNTA_PRUEBA, rol=ROL_PRUEBA, top_k=5)

print(f"Pregunta: {PREGUNTA_PRUEBA}")
print(f"Rol: {ROL_PRUEBA}\n")

for numero, c in enumerate(candidatos, start=1):
    print(f"{numero}. {c['fragmento_id']}")
    print(f"   Título: {c['titulo']}")
    print(f"   Clasificación: {c['clasificacion']}")
    print(f"   Similitud: {c.get('similitud', 0):.4f}")
    if "puntuacion_reranking" in c:
        print(f"   Reranking: {c['puntuacion_reranking']:.4f}")
    print()

Pregunta: ¿Cuál es el plazo para reclamar un faltante y desde cuándo se cuenta?
Rol: operador

1. POL-REC-001_v1.0_s7_p1
   Título: 7. [PÚBLICO] Plazos de presentación y evidencia por tipo
   Clasificación: publico
   Similitud: 0.7440
   Reranking: -2.2030

2. POL-REC-001_v1.0_s7_p2
   Título: 7. [PÚBLICO] Plazos de presentación y evidencia por tipo
   Clasificación: publico
   Similitud: 0.7309
   Reranking: -3.2613

3. POL-REC-001_v1.0_s17
   Título: 17. [PÚBLICO] Casos fuera de plazo, excepciones y contingencias
   Clasificación: publico
   Similitud: 0.7146
   Reranking: -4.2816

4. POL-REC-001_v1.0_s5
   Título: 5. [PÚBLICO] Datos para abrir un reclamo
   Clasificación: publico
   Similitud: 0.6908
   Reranking: -5.3796

5. POL-REC-001_v1.0_s10
   Título: 10. [PÚBLICO] Reglas particulares de entrega
   Clasificación: publico
   Similitud: 0.7115
   Reranking: -7.3037



## 3. Consulta completa (recuperación + reranking + generación)

Ejecuta el pipeline end-to-end y muestra la respuesta generada por Gemini junto con las fuentes utilizadas.

In [4]:
resultado = rag.consultar(PREGUNTA_PRUEBA, rol=ROL_PRUEBA, top_k=10, top_contexto=3)

print("RESPUESTA:\n")
print(resultado["respuesta"])
print("\n--- FUENTES ---")
for numero, fuente in enumerate(resultado["fuentes"], start=1):
    print(f"{numero}. {fuente['fragmento_id']} — {fuente['titulo']}")

RESPUESTA:

El plazo para reclamar un faltante es de **hasta 2 días hábiles** y se cuenta **desde la recepción física** [F1]. El día del hecho inicial se considera día cero y el conteo empieza al día siguiente [F3].

* Documento: POL-REC-001, versión 1.0, sección 7 (chunks POL-REC-001_v1.0_s7_p1 y POL-REC-001_v1.0_s7_p2) [F1, F3].

--- FUENTES ---
1. POL-REC-001_v1.0_s7_p1 — 7. [PÚBLICO] Plazos de presentación y evidencia por tipo
2. POL-REC-001_v1.0_s9 — 9. [PÚBLICO] Procedimiento general
3. POL-REC-001_v1.0_s7_p2 — 7. [PÚBLICO] Plazos de presentación y evidencia por tipo


## 4. Consultas adicionales

Modifica la pregunta y el rol para probar otras consultas.

In [6]:
pregunta = "¿Cuándo puede un cliente solicitar la anulación de un reclamo?"
rol = "operador"

respuesta = rag.consultar(pregunta, rol=rol, top_k=10, top_contexto=3)
print(f"Pregunta: {pregunta}")
print(f"Rol: {rol}\n")
print(respuesta["respuesta"])

Pregunta: ¿Cuándo puede un cliente solicitar la anulación de un reclamo?
Rol: operador

Información insuficiente. Las fuentes proporcionadas no mencionan cuándo un cliente puede solicitar la anulación de un reclamo. Solo detallan plazos para solicitar revisión, reapertura de tickets incompletos y presentación de devoluciones comerciales, pero no contienen información sobre la anulación de reclamos [F1, F2, F3].
